## FieldLine OPM Digitization and Coregistration

In the **FieldLine OPM system**, headshape points and fiducial points are not automatically stored in the `.fif` file. Instead, digitization is performed using a **Polhemus electromagnetic scanner** together with the **Brainstorm** software. This requires the `.fif`-file to updated as we will demonstrate here. 

### Digitization procedure
The digitization is performed twice:

1. **Outside the MSR (without helmet):**  
   - Headshape points are recorded.  
   - Fiducials (Nasion, LPA, RPA) are recorded.  

2. **Inside the MSR (with helmet):**  
   - Helmet reference points are recorded.  
   - The fiducial points are recorded again (the same points as recorded outside the MSR).  

### Common points
*** ARNAB; rewrite below for clarity (is it the 2 x 3 fiducial points? ****
- The **first six points** in the head digitization (outside) are the **common points** recorded in both sessions (later averaged).  
- These points are stored in the **NA, LPA, RPA fields** in the helmet file.  


*** ARNAB: this belongs under File overview ***
- **HS file (Headshape):** contains headshape and fiducial points.  
- **HR file (Helmet):** contains helmet reference points and the three common points (averaged).  

### Transformation Process

*** ARNAB: are the common points the fiducial points? ***
1. Compute a **helmet-to-head affine transform** using the common points.  
2. Apply this transform to re-reference the helmet data into the **head coordinate frame**.  
3. **Re-reference sensor locations** (originally defined in the **device coordinate frame**) into the **head coordinate frame**:  
   - This is done by an affine transform anchored at the helmet reference points.  
   - The reference point values are known in the **device coordinate frame**.  
4. Once everything is aligned in head coordinates, create a **new `.fif`-file** (copy of the original) with:  
   - Updated sensor locations  
   - Headshape points  
   - Fiducials


## Introduction

Load the following modules necessary for further processing

In [1]:
import mne
import numpy as np
import pandas as pd
import os
import os.path as op
import warnings
import inspect
from mne_bids import (
    BIDSPath,
    make_dataset_description,
    print_dir_tree,
    read_raw_bids,
    write_raw_bids
)
from mne.transforms import apply_trans
from mne.coreg import fit_matched_points
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D


# FLutils.py are a set of support functions for the transformations
from FLutils import (
    load_pos_file,
    compute_avg_fiducials,
    als_to_ras_f,
    als_to_ras,
    comp_avg_fiducials,
    extract_fake_fiducials,
    level_and_average_dig_points
)

warnings.filterwarnings('ignore')



*** ARBAN: please include full overview ***

### File overview
The chapter relies on the input file (embedded in the BIDS structure)

```
<BIDS_ROOT>/sub-01/ses-01/meg/sub-01_ses-01_task-SpAtt_run-01_meg.fif
P5EA_20250308_01HS
P5EA_20250308_01HR
```

## Loading the BIDS Data

In [2]:
subject = '01'  
session = '01'  
task = 'SpAtt'
run = '01'  
meg_suffix = 'meg'
meg_extension = '.fif'
events_suffix = 'events'
events_extension = '.tsv'

# data_path='E:/Sub_OJ_new'
data_path = '/Users/o.jensen@bham.ac.uk/Data/FieldLineUoB'


bids_root = op.join(data_path, "Fieldline_Spatt_BIDS")
bids_path = BIDSPath(subject=subject, session=session,
                     task=task, run=run, suffix=meg_suffix, 
                     extension=meg_extension, root=bids_root)

print(bids_path)

#*** ARNAB: is this correct? ***
head_file = op.join(data_path, "Digitization/P5EA_20250308_01HS.pos");
helmet_file= op.join(data_path, "Digitization/P5EA_20250308_01HR.pos");

df_excel_file = op.join(data_path, "Digitization/opm_channels_coordinates.xlsx");

updated_meg_fif_path = op.join(bids_root, "sub-01/ses-01/meg/updated_final.fif")
print(updated_meg_fif_path)
                               
#"E:/Sub_OJ_new/Fieldline_Spatt_BIDS/sub-01/ses-01/meg/updated_final.fif"


#*** ARNAB : move the load to own section - this one is about file def ***
raw = read_raw_bids(bids_path=bids_path, verbose=False,extra_params={'preload':True})

/Users/o.jensen@bham.ac.uk/Data/FieldLineUoB/Fieldline_Spatt_BIDS/sub-01/ses-01/sub-01_ses-01_task-SpAtt_run-01_meg.fif
/Users/o.jensen@bham.ac.uk/Data/FieldLineUoB/Fieldline_Spatt_BIDS/sub-01/ses-01/meg/updated_final.fif


## Data loading and units

The function loads both the **helmet (HR)** and **head (HS)** digitization data. Because the coordinates are in **centimeters (cm)**, they are first converted to **meters (m)** before any further processing. This ensures consistency with analysis tools (e.g., MNE-Python), which expect coordinates in meters.


In [3]:
head_dig_points, head_fiducials = load_pos_file(head_file,np)
helmet_dig_points, helmet_fiducials = load_pos_file(helmet_file,np)

print(f"Loaded {len(head_dig_points)} head points and {len(helmet_dig_points)} helmet points.")
print("Head Fiducials:", head_fiducials)
print("Helmet Fiducials:", helmet_fiducials)

Loaded 498 head points and 16 helmet points.
Head Fiducials: {'NA': array([[ 0.09707636,  0.0004379 ,  0.00067454],
       [ 0.0952321 , -0.0004379 , -0.00067454]]), 'LPA': array([[ 0.00333986,  0.07587232,  0.00036244],
       [ 0.00215557,  0.08113519, -0.00036244]]), 'RPA': array([[-0.00271597, -0.07824513, -0.00093321],
       [-0.00277946, -0.07876237,  0.00093321]])}
Helmet Fiducials: {'NA': array([[ 0.03762235, -0.00096262, -0.00039603],
       [ 0.03801041,  0.00096262,  0.00039603]]), 'LPA': array([[ 0.00077279,  0.04647964,  0.00038936],
       [-0.00305244,  0.04762977, -0.00038936]]), 'RPA': array([[ 0.00120447, -0.04726171, -0.00014416],
       [ 0.00107518, -0.0468477 ,  0.00014416]])}


## Averaging fiducial points

Each fiducial point (Nasion, LPA, RPA) was measured twice within the **helmet** file and twice within the **head** file.  For each file separately, the duplicate measurements are averaged to obtain a single representative point per fiducial.


In [4]:
avg_head_fiducials = compute_avg_fiducials(head_fiducials,np)
avg_helmet_fiducials = compute_avg_fiducials(helmet_fiducials,np)
print("Averaged Head Fiducials:", avg_head_fiducials)
print("Averaged Helmet Fiducials:", avg_helmet_fiducials)

Averaged Head Fiducials: {'NA': array([0.09615423, 0.        , 0.        ]), 'LPA': array([0.00274771, 0.07850375, 0.        ]), 'RPA': array([-0.00274771, -0.07850375,  0.        ])}
Averaged Helmet Fiducials: {'NA': array([0.03781638, 0.        , 0.        ]), 'LPA': array([-0.00113982,  0.0470547 ,  0.        ]), 'RPA': array([ 0.00113982, -0.0470547 ,  0.        ])}


## Conversion of Coordinate System: CTF to Device

*** ARNAB:  what does RAS mean? ***


The Polhemus system records digitization data in the **CTF coordinate frame**  (+X forward, +Y left, +Z up), (CTF refers to a cryo-MEG system; the terminoloy here is historical) while the device coordinate system is typically defined  in an **RAS frame** (+X right, +Y anterior, +Z superior).  To unify these representations, the digitized data are rotated (via a coordinate  transform) from the CTF frame into the device (RAS) frame.


In [5]:
ras_head = als_to_ras(head_dig_points,np)
ras_helmet = als_to_ras(helmet_dig_points,np)
als_array1 = np.stack(list(head_fiducials.values()))
als_array2 = np.stack(list(helmet_fiducials.values()))
ras_head_array = als_to_ras_f(als_array1,np)
ras_helmet_array = als_to_ras_f(als_array2,np)
ras_head_fiducials = {
    key: val for key, val in zip(head_fiducials.keys(), ras_head_array)
}
ras_helmet_fiducials = {
    key: val for key, val in zip(helmet_fiducials.keys(), ras_helmet_array)
}

## Averaging fiducial points after coordinate transformation

Once the coordinate system has been rotated into the device (RAS) frame, the fiducial points (Nasion, LPA, RPA) are averaged again to obtain  a single consistent set of fiducials in the new coordinate system.


In [6]:

ras_head_fiducials=comp_avg_fiducials(ras_head_array,np)
ras_helmet_fiducials=comp_avg_fiducials(ras_helmet_array,np)
ras_avg_head_fiducials = {key: ras_head_fiducials[i] for i, key in enumerate(head_fiducials)}
ras_avg_helmet_fiducials = {key: ras_helmet_fiducials[i] for i, key in enumerate(helmet_fiducials)}

print(ras_avg_head_fiducials)
print(ras_avg_helmet_fiducials)

{'NA': array([0.        , 0.09615423, 0.        ]), 'LPA': array([-0.07850375,  0.00274771,  0.        ]), 'RPA': array([ 0.07850375, -0.00274771,  0.        ])}
{'NA': array([0.        , 0.03781638, 0.        ]), 'LPA': array([-0.0470547 , -0.00113982,  0.        ]), 'RPA': array([0.0470547 , 0.00113982, 0.        ])}


## Extract fake fiducials from head digitization

As mentioned earlier, three common points ***ARNAB: is this the fiducials? - if so, call them that *** are recorded during both head and helmet digitizations.  In the **head digitization file**, these points appear as the **first six entries**  (each point measured twice). Although they are not true anatomical fiducials,  they act as reference fiducials in the helmet coordinate system.  We refer to these as **fake fiducials**, designated as **FNA**, **FLPA**, and **FRPA**.  The function below extracts these points from the head digitization file  and computes their average values.


In [7]:
ras_fake_fiducials, ras_avg_fake_fiducials = extract_fake_fiducials(ras_head,np)

print("Extracted ras Fake Fiducials:", ras_fake_fiducials)
print("Averaged ras Fake Fiducials:", ras_avg_fake_fiducials)

Extracted ras Fake Fiducials: {'FNA': array([[ 0.00133957,  0.09358993,  0.00059375],
       [ 0.00235911,  0.09776012, -0.00120527]]), 'FLPA': array([[-0.04540028,  0.07469059, -0.03607888],
       [-0.04748642,  0.07823816, -0.03790051]]), 'FRPA': array([[ 0.04217455,  0.07606987, -0.03856865],
       [ 0.03530665,  0.07453986, -0.03447446]])}
Averaged ras Fake Fiducials: {'FNA': array([ 0.00184934,  0.09567503, -0.00030576]), 'FLPA': array([-0.04644335,  0.07646438, -0.03698969]), 'FRPA': array([ 0.0387406 ,  0.07530487, -0.03652155])}


## Helmet-to-head transformation

At this stage, we have the **common points** represented in both coordinate frames:

- In the **head digitization file**, the common points are available in the head reference frame.  
- In the **helmet digitization file**, the same points are stored under the NA, LPA, and RPA fields.  

These two corresponding point sets are used to compute a **homogeneous transformation matrix**  via an affine transformation. This matrix is then applied to re-reference all helmet points,  including the **8 helmet reference points**, into the **head coordinate frame**.


In [8]:
target_points = np.array([ras_avg_fake_fiducials["FNA"], ras_avg_fake_fiducials["FLPA"], ras_avg_fake_fiducials["FRPA"]])  # Fake fiducials
source_points = np.array([ras_avg_helmet_fiducials["NA"], ras_avg_helmet_fiducials["LPA"], ras_avg_helmet_fiducials["RPA"]])  # Helmet fiducials
helmet_to_head_transformation_matrix = mne.coreg.fit_matched_points(source_points,target_points)
   

transformed_helmet_fiducials_head = mne.transforms.apply_trans(helmet_to_head_transformation_matrix, np.array([ras_avg_helmet_fiducials["NA"], ras_avg_helmet_fiducials["LPA"], ras_avg_helmet_fiducials["RPA"]]))
transformed_helmet_reference_points = mne.transforms.apply_trans(helmet_to_head_transformation_matrix, ras_helmet)




print("Helmet to Head Transformation Matrix:\n", helmet_to_head_transformation_matrix)
print("Transformed Helmet Fiducials (Should Match Real Fiducials):\n", transformed_helmet_fiducials_head)
print("Transformed Helmet Reference Points:\n", transformed_helmet_reference_points)

Helmet to Head Transformation Matrix:
 [[ 0.99869572  0.0489314  -0.01457969 -0.00256794]
 [-0.03622006  0.47771424 -0.87776832  0.07645962]
 [-0.03598551  0.87715154  0.47886346 -0.03566257]
 [ 0.          0.          0.          1.        ]]
Transformed Helmet Fiducials (Should Match Real Fiducials):
 [[-0.00071753  0.09452504 -0.00249187]
 [-0.04961704  0.07761943 -0.03496908]
 [ 0.04448116  0.0752998  -0.03635605]]
Transformed Helmet Reference Points:
 [[ 0.0724548   0.10889625  0.01488331]
 [ 0.10781614  0.05006922 -0.01832228]
 [ 0.10734543 -0.00870529  0.0351853 ]
 [ 0.04536655 -0.09897784 -0.05363395]
 [-0.05048801  0.11428386  0.02135902]
 [-0.09344582  0.06552596 -0.02028901]
 [-0.12146322  0.00311931  0.03761382]
 [-0.07344746 -0.08474021 -0.05644068]
 [ 0.07326281  0.10939671  0.01441803]
 [ 0.10837876  0.05041624 -0.01860439]
 [ 0.10766703 -0.00869492  0.03548486]
 [ 0.04642741 -0.09868348 -0.05342758]
 [-0.0505606   0.11557787  0.01865561]
 [-0.094388    0.06493449 -0.016

## Plotting digitization points

As a sanity check, we will plot both the **original fiducial points**  and the **transformed fiducial points**.  

The goal is to verify that, after applying the transformation,  the **fake fiducials** align correctly between the head and helmet coordinate frames.


In [9]:
%matplotlib qt

#def plot_head_points_and_fiducials(transformed_helmet_fiducials_head, transformed_helmet_reference_points, ras_avg_fake_fiducials, ras_avg_head_fiducials,ras_helmet):

# Create 3D plot
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
points=np.round(transformed_helmet_fiducials_head,8)

ax.scatter(points[:, 0], points[:, 1], points[:, 2], c='red', label='Transformed_helmet_fiducials', s=80)

# Plot transformed head points
ax.scatter(transformed_helmet_reference_points[:, 0], transformed_helmet_reference_points[:, 1],transformed_helmet_reference_points[:, 2], c='green', label='Transformed Helmet Reference Points', s=30)
ax.scatter(ras_helmet[:, 0], ras_helmet[:, 1],ras_helmet[:, 2], c='blue', label='Helmet Reference Points before Transformation', s=50)

# Plot Fake fiducials fiducials (FNA, FLPA, FRPA) recorded during helmet digitization
ax.scatter(ras_avg_fake_fiducials["FNA"][0], ras_avg_fake_fiducials["FNA"][1], ras_avg_fake_fiducials["FNA"][2], c='red', label='FNA', s=50)
ax.scatter(ras_avg_fake_fiducials["FLPA"][0], ras_avg_fake_fiducials["FLPA"][1], ras_avg_fake_fiducials["FLPA"][2], c='orange', label='FLPA', s=50)
ax.scatter(ras_avg_fake_fiducials["FRPA"][0], ras_avg_fake_fiducials["FRPA"][1], ras_avg_fake_fiducials["FRPA"][2], c='yellow', label='FRPA', s=50)

# Plot Real fiducials (NA, LPA, RPA)
ax.scatter(ras_avg_head_fiducials["NA"][0], ras_avg_head_fiducials["NA"][1], ras_avg_head_fiducials["NA"][2], c='purple', label='NA', s=50)
ax.scatter(ras_avg_head_fiducials["LPA"][0], ras_avg_head_fiducials["LPA"][1], ras_avg_head_fiducials["LPA"][2], c='pink', label='LPA', s=50)
ax.scatter(ras_avg_head_fiducials["RPA"][0], ras_avg_head_fiducials["RPA"][1], ras_avg_head_fiducials["RPA"][2], c='cyan', label='RPA', s=50)

# Set labels
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
    
ax.set_title('Aignment checking')
ax.legend()

plt.show()


#plot_head_points_and_fiducials(transformed_helmet_fiducials_head, transformed_helmet_reference_points, ras_avg_fake_fiducials, ras_avg_head_fiducials,ras_helmet)


<img src="Fig1.png" alt="Digitization Points" width="500"/>


## Retrieving helmet reference points

Next, we extract the coordinates of the reference points digitized on the FieldLine Beta2 helmet.  It is important that these points are retrieved in the **same sequence**  as they were scanned, so they can be directly matched with the  corresponding coordinates defined in the device frame.


In [10]:

ref_helmet_positions = level_and_average_dig_points(transformed_helmet_reference_points,helmet_dig_points,np)
print("Ref Helmet Positions:", ref_helmet_positions)

Ref Helmet Positions: {'A1': array([0.0728588 , 0.10914648, 0.01465067]), 'A2': array([ 0.10809745,  0.05024273, -0.01846333]), 'A3': array([ 0.10750623, -0.00870011,  0.03533508]), 'A4': array([ 0.04589698, -0.09883066, -0.05353077]), 'A5': array([-0.05052431,  0.11493087,  0.02000732]), 'A6': array([-0.09391691,  0.06523022, -0.01822214]), 'A7': array([-0.12157637,  0.0033537 ,  0.03711072]), 'A8': array([-0.07332509, -0.08456096, -0.05684756])}


## Original helmet reference point coordinates from FieldLine

The FieldLine Beta2 helmet comes with predefined reference point locations specified in the **device coordinate frame**.  These coordinates are provided in **millimeters (mm)**. Before further processing, they are retrieved and converted into **meters (m)** to ensure consistency with the rest of the analysis.


In [11]:
original_ref_points = np.array([
    [55.703, 119.485, -0.749],
    [92.105, 59.254, -28.708],
    [102.325, 0.221, 16.345],
    [58.503, -94.717, -69.944],
    [-55.703, 119.485, -0.749],
    [-92.105, 59.254, -28.708],
    [-102.325, 0.221, 16.345],
    [-58.503, -94.717, -69.944]
])
original_ref_points_meter=original_ref_points/1000  
original_labels = ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8']
original_ref_points_dict = {label: point for label, point in zip(original_labels, original_ref_points_meter)}

## Affine transformation for sensor re-referencing

An affine transformation (homogeneous) matrix is computed using the helmet reference point locations in both coordinate frames:  

- **Device coordinate frame** (from FieldLine-provided values)  
- **Head coordinate frame** (obtained previously from digitization)  

This transformation matrix is then applied to re-reference all OPM sensor locations from the **device coordinate frame** into the  **head coordinate frame**.


In [12]:
ref_helmet_array = np.array([ref_helmet_positions[label] for label in original_labels])
sensor_to_head_transformation = mne.coreg.fit_matched_points(original_ref_points_meter,ref_helmet_array,scale=True)
transformed_array = mne.transforms.apply_trans(sensor_to_head_transformation, original_ref_points_meter)
transformed_dict = {label: point for label, point in zip(original_labels, transformed_array)}

## OPM sensor locations from FieldLine

FieldLine provides the OPM sensor locations for the helmet in a `.mat` file.  These channel locations have been extracted from the file and saved in an  `.xls` format for convenience.  The coordinates are defined with respect to the **device coordinate frame** and are already given in **meters (m)**, so no unit conversion is required. Thereafter, the channel locations are re-referenced to Head co-ordinate system using the previously derived sensor_to_head_transformation matrix.


In [13]:
df = pd.read_excel(df_excel_file)

coordinates = df[["x_m", "y_m", "z_m"]].values.tolist()
meg_channel_locations = dict(zip(df["channel"], coordinates))
#print(meg_channel_locations) ## Optional

meg_labels = list(meg_channel_locations.keys())
meg_array  = np.array(list(meg_channel_locations.values()), dtype=float)
#transformed_meg_array = mne.transforms.apply_trans(sensor_to_head_transformation, meg_array)
#transformed_meg_dict = {lab: pt for lab, pt in zip(meg_labels, transformed_meg_array)}

## Plotting the reference points

Before transforming the OPM channel locations, we first plot all reference points.  This serves as a sanity check to visually confirm that the computed  transformation is correct.


In [15]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
    
for label, point in original_ref_points_dict.items():
    ax.scatter(point[0], point[1], point[2], color='red',s=50, label=f"Original Ref Points" if label == 'A1' else "")
    ax.text(point[0], point[1], point[2], label, fontsize=10, color='red')
    
for label, point in ref_helmet_positions.items():
    ax.scatter(point[0], point[1], point[2], color='blue', label=f"Helmet Reference before Transformation" if label == 'A1' else "")
    ax.text(point[0], point[1], point[2], label, fontsize=10, color='blue')
    
for label, point in transformed_dict.items():
    ax.scatter(point[0], point[1], point[2], color='green', label=f"Transformed Helmet Ref points " if label == 'A1' else "")
    ax.text(point[0], point[1], point[2], label, fontsize=10, color='green')
ax.scatter(meg_array[:, 0], meg_array[:, 1], meg_array[:, 2], c='y', s=20)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title("Helmet Digitization Points: Original, Measured, and Transformed")
ax.legend()
plt.show()



<img src="Fig2.png" alt="Referenc points" width="800"/>

## Transform the Sensor Co-ordinates 
Here the OPM channel locations are transformed to the head cordinate.

In [16]:
transformed_meg_array = mne.transforms.apply_trans(sensor_to_head_transformation, meg_array)

## Plotting headshape points with transformed sensor locations

As a sanity check, we plot the **headshape points** together with the  **re-referenced sensor locations**.  This visualization helps verify that the helmet and head coordinate  frames are properly aligned after transformation.


In [17]:
def plot_head_points_and_fiducials(ras_head, meg_channel_locations,transformed_meg_array):
    if isinstance(meg_channel_locations, dict):
        meg_coords = np.array(list(meg_channel_locations.values()))
    else:
        meg_coords = np.asarray(meg_channel_locations)
    # Create 3D plot
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Plot transformed head points
    ax.scatter(ras_head[:, 0], ras_head[:, 1], ras_head[:, 2],c='green', label='Head Points', s=30)
    #ax.scatter(transformed_array[:, 0], transformed_array[:, 1], transformed_array[:, 2],c='blue', label='Reference Points', s=60)
    # Plot MEG channel locations
    #ax.scatter(meg_coords[:, 0], meg_coords[:, 1], meg_coords[:, 2], c='blue', label='MEG Channel Locations', s=30)
     # Plot MEG channel locations
    ax.scatter(transformed_meg_array[:, 0], transformed_meg_array[:, 1], transformed_meg_array[:, 2],c='red', label='MEG Channel Locations Transformed', s=30)
    # Set labels
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title('Head Shape Points Alignment with Helmet Fiducials')
    ax.legend()
    plt.show()
%matplotlib qt
plot_head_points_and_fiducials(ras_head, meg_channel_locations,transformed_meg_array)

<img src="Fig4.png" alt="Digitization Points" width="800"/>

## Updating channel locations in the raw object

At this step, we update the channel location information in the raw object.  Instead of directly overwriting the channel coordinates with the transformed  values, we store the **original channel locations** and update the  `dev_head_t` matrix inside the raw object.  By storing our previously computed sensor-to-head transformation matrix in  `dev_head_t`, the sensor locations will be correctly transformed into the  head coordinate frame during GUI coregistration in the next script. Only the channels present in the current experiment will be updated.


In [18]:
raw1=raw.copy()
def replace_channel_locations(raw1, meg_channel_locations):
    updated_channels = []
    skipped_channels = []

    for ch_idx, ch_name in enumerate(raw.ch_names):
        match = next((key for key in meg_channel_locations if ch_name.startswith(key)), None)
        if match:
            raw.info['chs'][ch_idx]['loc'][:3] = np.array(meg_channel_locations[match])
            updated_channels.append(ch_name)
        else:
            skipped_channels.append(ch_name)

    print(f"Updated {len(updated_channels)} channels:")
    for name in updated_channels:
        print(f"  - {name}")

    print(f"\nSkipped {len(skipped_channels)} channels:")
    for name in skipped_channels:
        print(f"  - {name}")

meg_channel_locations_new = {name: pos[:3] for name, pos in zip(meg_labels, meg_array)}
replace_channel_locations(raw1, meg_channel_locations_new)


Updated 68 channels:
  - L102_bz-s73
  - L104_bz-s80
  - L106_bz-s84
  - L108_bz-s77
  - L110_bz-s76
  - L112_bz-s44
  - L114_bz-s50
  - L116_bz-s37
  - L201_bz-s86
  - L205_bz-s87
  - L207_bz-s75
  - L209_bz-s52
  - L210_bz-s65
  - L211_bz-s46
  - L213_bz-s48
  - L215_bz-s47
  - L302_bz-s83
  - L304_bz-s71
  - L306_bz-s67
  - L307_bz-s42
  - L309_bz-s68
  - L310_bz-s45
  - L312_bz-s49
  - L401_bz-s85
  - L403_bz-s81
  - L405_bz-s82
  - L409_bz-s43
  - L503_bz-s79
  - L505_bz-s74
  - L507_bz-s70
  - L509_bz-s51
  - L603_bz-s78
  - L605_bz-s72
  - L607_bz-s9
  - R102_bz-s4
  - R104_bz-s11
  - R106_bz-s3
  - R108_bz-s12
  - R110_bz-s1
  - R112_bz-s39
  - R114_bz-s35
  - R116_bz-s36
  - R201_bz-s6
  - R203_bz-s7
  - R205_bz-s8
  - R207_bz-s10
  - R209_bz-s38
  - R210_bz-s66
  - R211_bz-s34
  - R213_bz-s32
  - R215_bz-s25
  - R302_bz-s13
  - R304_bz-s5
  - R306_bz-s69
  - R307_bz-s2
  - R309_bz-s27
  - R310_bz-s19
  - R312_bz-s33
  - R401_bz-s30
  - R403_bz-s29
  - R405_bz-s40
  - R407_bz-

## Storing the sensor-to-head transformation

The computed sensor-to-head transformation matrix is stored in:

`raw.info['dev_head_t']`

This ensures that the sensor locations can be automatically transformed  
into the head coordinate frame during later processing and coregistration.


In [19]:
from mne.transforms import Transform
channel_locs_n = np.array([raw1.info['chs'][ch_idx]['loc'][:3] for ch_idx in range(len(raw1.ch_names))])
transformation1 = Transform('meg', 'head', sensor_to_head_transformation)
raw1.info['dev_head_t'] = transformation1
### Print the matrix for verification
raw1.info['dev_head_t']

<Transform | MEG device->head>
[[ 1.03387272  0.09893145  0.00343189 -0.00263521]
 [-0.09886901  1.03376505 -0.01570835 -0.0032465 ]
 [-0.0049122   0.01531014  1.03847652  0.01624548]
 [ 0.          0.          0.          1.        ]]

## Including head digitization and fiducial points

Finally, we accumulate all digitization information (headshape points and fiducials) into a montage object.  This montage is then stored inside the `.fif` file to ensure that the head  geometry and fiducials are available for coregistration and subsequent analysis

In [20]:
meg_picks = mne.pick_types(raw1.info, meg=True, stim=False)
channel_locs1 = np.array([raw1.info['chs'][i]['loc'][:3] for i in meg_picks])
channel_pos_dict = {ch_name: pos for ch_name, pos in zip(raw1.ch_names, channel_locs1)}
montage = mne.channels.make_dig_montage(
    hsp=ras_head,                                # headshape points
    ch_pos=channel_pos_dict,
    lpa=ras_avg_head_fiducials.get('LPA'),  # Left preauricular fiducial (Meters)
    nasion=ras_avg_head_fiducials.get('NA'),  # Nasion fiducial (Meters)
    rpa=ras_avg_head_fiducials.get('RPA'),
    coord_frame="head"
)

raw1.set_montage(montage, on_missing="warn")

<Raw | sub-01_ses-01_task-SpAtt_run-01_meg.fif, 69 x 2408520 (2408.5 s), ~1.24 GiB, data loaded>

## Plotting the montage

To verify the montage, we plot it and visually inspect the  fiducials, headshape points, and sensor locations to confirm  that everything is correctly aligned.

In [21]:
fig = montage.plot(show=False)  # prevent auto-show so we can edit
fig.set_size_inches(20, 20)
plt.show()

<img src="Fig3.png" alt="Digitization Points" width="800"/>

## Save the updated file

Finally the the updated raw1 file with newly created montage will be saved in a fif file. Please note this updated fif file should be used during GUI Co registration.

In [22]:
raw1.save(updated_meg_fif_path, overwrite=True)

Overwriting existing file.
Writing /Users/o.jensen@bham.ac.uk/Data/FieldLineUoB/Fieldline_Spatt_BIDS/sub-01/ses-01/meg/updated_final.fif
Closing /Users/o.jensen@bham.ac.uk/Data/FieldLineUoB/Fieldline_Spatt_BIDS/sub-01/ses-01/meg/updated_final.fif
[done]


[PosixPath('/Users/o.jensen@bham.ac.uk/Data/FieldLineUoB/Fieldline_Spatt_BIDS/sub-01/ses-01/meg/updated_final.fif')]